In [ ]:
# ============================================================
# SIGN LANGUAGE ALPHABET RECOGNITION
# RESNET18 + GOOGLE COLAB WEBCAM + TAMIL TRANSLATION
# COMPLETE CONSOLIDATED CODE
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q deep-translator


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import copy
import warnings
import time
import base64

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import cv2

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader

from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix, classification_report

from IPython.display import display, Javascript

from google.colab.output import eval_js

from deep_translator import GoogleTranslator


# Silence PIL warning
warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency"
)


print("\nLibraries imported successfully")


# ============================================================
# 3. CHECK GPU
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

    # Helps when all images have same size
    torch.backends.cudnn.benchmark = True

else:

    print(
        "GPU is not available."
        " Training will use CPU."
    )


# ============================================================
# 4. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully")


# ============================================================
# 5. DATASET PATH
# ============================================================

drive_dataset = (
    "/content/drive/MyDrive/"
    "final_project_sign_dataset"
)

local_dataset = (
    "/content/final_project_sign_dataset"
)


print("\nChecking dataset...")


# ============================================================
# 6. COPY DATASET FROM DRIVE TO COLAB LOCAL STORAGE
# ============================================================

if not os.path.exists(local_dataset):

    print(
        "\nCopying dataset from Google Drive "
        "to Colab local storage..."
    )

    print(
        "This may take some time the first time."
    )

    !cp -r "/content/drive/MyDrive/final_project_sign_dataset" "/content/"

    print(
        "\nDataset copied successfully!"
    )

else:

    print(
        "\nDataset already exists in "
        "Colab local storage."
    )




In [ ]:
# ============================================================
# 7. TRAIN AND TEST DIRECTORIES
# ============================================================

data_dir = local_dataset

train_dir = os.path.join(
    data_dir,
    "train"
)

test_dir = os.path.join(
    data_dir,
    "test"
)


print(
    "\nDataset directory:",
    data_dir
)

print(
    "Train directory:",
    train_dir
)

print(
    "Test directory:",
    test_dir
)


# ============================================================
# 8. CHECK DATASET
# ============================================================

if not os.path.exists(train_dir):

    raise FileNotFoundError(
        f"Train directory not found: {train_dir}"
    )


if not os.path.exists(test_dir):

    raise FileNotFoundError(
        f"Test directory not found: {test_dir}"
    )


print("\nDataset structure is valid!")


# ============================================================
# 9. CONFIGURATION
# ============================================================

image_size = 224

batch_size = 64

epochs = 3

learning_rate = 0.001

num_workers = 2

model_path = (
    "/content/drive/MyDrive/"
    "assignment_final.pth"
)


print("\n========== CONFIGURATION ==========")

print(
    "Image size     :",
    image_size
)

print(
    "Batch size     :",
    batch_size
)

print(
    "Number epochs  :",
    epochs
)

print(
    "Learning rate  :",
    learning_rate
)

print(
    "Workers        :",
    num_workers
)


# ============================================================
# 10. MIXED PRECISION
# ============================================================

use_amp = torch.cuda.is_available()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

print(
    "Mixed Precision:",
    use_amp
)


# ============================================================
# 11. IMAGE TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (image_size, image_size)
    ),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


test_transform = transforms.Compose([

    transforms.Resize(
        (image_size, image_size)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# ============================================================
# 12. LOAD DATASETS
# ============================================================

print("\nLoading training images...")

train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

print(
    "Training dataset loaded."
)


print("\nLoading test images...")

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=test_transform
)

print(
    "Test dataset loaded."
)


# ============================================================
# 13. DATASET INFORMATION
# ============================================================

class_names = train_dataset.classes

num_classes = len(class_names)


print(
    "\n========== DATASET INFORMATION =========="
)

print(
    "Number of classes:",
    num_classes
)

print(
    "Training images:",
    len(train_dataset)
)

print(
    "Test images:",
    len(test_dataset)
)


print("\nClasses:")

for i, class_name in enumerate(class_names):

    print(
        i,
        ":",
        class_name
    )


# Check classes

if train_dataset.classes != test_dataset.classes:

    raise ValueError(
        "Training and test classes do not match!"
    )


# ============================================================
# 14. DATALOADERS
# ============================================================

use_pin_memory = (
    device.type == "cuda"
)


train_loader = DataLoader(

    train_dataset,

    batch_size=batch_size,

    shuffle=True,

    num_workers=num_workers,

    pin_memory=use_pin_memory,

    persistent_workers=(
        True if num_workers > 0 else False
    ),

    prefetch_factor=(
        2 if num_workers > 0 else None
    )
)


test_loader = DataLoader(

    test_dataset,

    batch_size=batch_size,

    shuffle=False,

    num_workers=num_workers,

    pin_memory=use_pin_memory,

    persistent_workers=(
        True if num_workers > 0 else False
    ),

    prefetch_factor=(
        2 if num_workers > 0 else None
    )
)


print(
    "\nTraining batches:",
    len(train_loader)
)

print(
    "Testing batches:",
    len(test_loader)
)


# ============================================================
# 15. TEST IMAGE LOADING SPEED
# ============================================================

print(
    "\nTesting image loading..."
)

start_time = time.time()

images, labels = next(
    iter(train_loader)
)

loading_time = (
    time.time() - start_time
)

print(
    "First batch loading time:",
    round(loading_time, 2),
    "seconds"
)

print(
    "Batch image shape:",
    images.shape
)

print(
    "Batch labels shape:",
    labels.shape
)


# ============================================================
# 16. DISPLAY SAMPLE IMAGES
# ============================================================

def show_samples(
    dataset,
    class_names,
    number_of_images=8
):

    indices = np.random.choice(

        len(dataset),

        min(
            number_of_images,
            len(dataset)
        ),

        replace=False
    )

    plt.figure(
        figsize=(16, 8)
    )

    for i, index in enumerate(indices):

        image, label = dataset[index]

        image = (
            image.numpy()
            .transpose((1, 2, 0))
        )

        mean = np.array(
            [
                0.485,
                0.456,
                0.406
            ]
        )

        std = np.array(
            [
                0.229,
                0.224,
                0.225
            ]
        )

        image = (
            std * image + mean
        )

        image = np.clip(
            image,
            0,
            1
        )

        plt.subplot(
            2,
            4,
            i + 1
        )

        plt.imshow(image)

        plt.title(
            class_names[label]
        )

        plt.axis("off")

    plt.tight_layout()

    plt.show()


show_samples(
    train_dataset,
    class_names,
    number_of_images=8
)


# ============================================================
# 17. LOAD RESNET18
# ============================================================

print(
    "\nLoading pretrained RESNET18..."
)


weights = models.ResNet18_Weights.DEFAULT


model = models.resnet18(
    weights=weights
)


# Replace final classification layer

number_of_features = (
    model.fc.in_features
)


model.fc = nn.Linear(
    number_of_features,
    num_classes
)


# Move model to GPU

model = model.to(device)


print(
    "ResNet18 loaded successfully."
)

print(
    "Model device:",
    next(model.parameters()).device
)


# ============================================================
# 18. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()


# ============================================================
# 19. OPTIMIZER
# ============================================================

optimizer = optim.AdamW(

    model.parameters(),

    lr=learning_rate,

    weight_decay=1e-4
)


# ============================================================
# 20. LEARNING RATE SCHEDULER
# ============================================================

scheduler = optim.lr_scheduler.StepLR(

    optimizer,

    step_size=5,

    gamma=0.1
)


print(
    "\nLoss, optimizer and scheduler ready."
)


# ============================================================
# 21. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(

    model,
    data_loader,
    criterion,
    optimizer,
    scaler,
    epoch

):

    model.train()

    running_loss = 0.0

    correct = 0

    total = 0

    total_batches = len(
        data_loader
    )


    for batch_idx, (
        images,
        labels
    ) in enumerate(data_loader):


        # Move images to GPU

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        # Clear gradients

        optimizer.zero_grad(
            set_to_none=True
        )


        # Forward pass

        with torch.autocast(

            device_type="cuda",

            dtype=torch.float16,

            enabled=use_amp

        ):

            outputs = model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )


        # Backpropagation

        scaler.scale(
            loss
        ).backward()


        scaler.step(
            optimizer
        )

        scaler.update()


        # Statistics

        running_loss += (
            loss.item()
        )


        predictions = outputs.argmax(
            dim=1
        )


        total += labels.size(0)


        correct += (
            predictions == labels
        ).sum().item()


        # Progress

        if (

            batch_idx == 0

            or

            (batch_idx + 1) % 20 == 0

            or

            batch_idx + 1 == total_batches

        ):

            accuracy = (
                100.0 *
                correct /
                total
            )


            print(

                f"\rEpoch {epoch}/{epochs} | "

                f"Batch "
                f"{batch_idx + 1}/"
                f"{total_batches} | "

                f"Loss: "
                f"{loss.item():.4f} | "

                f"Accuracy: "
                f"{accuracy:.2f}%",

                end=""
            )


    print()


    epoch_loss = (
        running_loss /
        total_batches
    )


    epoch_accuracy = (
        100.0 *
        correct /
        total
    )


    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# 22. VALIDATION
# ============================================================

@torch.inference_mode()

def validate_model(

    model,
    data_loader,
    criterion

):

    model.eval()

    running_loss = 0.0

    correct = 0

    total = 0


    for images, labels in data_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        with torch.autocast(

            device_type="cuda",

            dtype=torch.float16,

            enabled=use_amp

        ):

            outputs = model(
                images
            )

            loss = criterion(
                outputs,
                labels
            )


        running_loss += (
            loss.item()
        )


        predictions = outputs.argmax(
            dim=1
        )


        total += labels.size(0)


        correct += (
            predictions == labels
        ).sum().item()


    epoch_loss = (
        running_loss /
        len(data_loader)
    )


    epoch_accuracy = (
        100.0 *
        correct /
        total
    )


    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# 23. TRAIN MODEL
# ============================================================

train_losses = []

val_losses = []

train_accuracies = []

val_accuracies = []


best_val_accuracy = 0.0


print(
    "\n========================================"
)

print(
    "STARTING TRAINING"
)

print(
    "========================================"
)


for epoch in range(
    1,
    epochs + 1
):


    print(
        f"\n========== "
        f"EPOCH {epoch}/{epochs} "
        f"=========="
    )


    # Training

    train_loss, train_accuracy = (
        train_one_epoch(

            model,

            train_loader,

            criterion,

            optimizer,

            scaler,

            epoch

        )
    )


    # Validation

    print(
        "Running validation..."
    )


    val_loss, val_accuracy = (
        validate_model(

            model,

            test_loader,

            criterion

        )
    )


    # Store results

    train_losses.append(
        train_loss
    )

    val_losses.append(
        val_loss
    )

    train_accuracies.append(
        train_accuracy
    )

    val_accuracies.append(
        val_accuracy
    )


    # Scheduler

    scheduler.step()


    # Print results

    print(
        f"Train Loss      : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Accuracy  : "
        f"{train_accuracy:.2f}%"
    )

    print(
        f"Validation Loss : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Acc. : "
        f"{val_accuracy:.2f}%"
    )

    print(
        f"Learning Rate   : "
        f"{optimizer.param_groups[0]['lr']:.6f}"
    )


    # Save best model

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = (
            val_accuracy
        )


        torch.save(

            {

                "model_state_dict":
                    model.state_dict(),

                "class_names":
                    class_names,

                "num_classes":
                    num_classes,

                "image_size":
                    image_size

            },

            model_path

        )


        print(
            "\nBest model saved!"
        )

        print(
            "Best validation accuracy:",
            f"{best_val_accuracy:.2f}%"
        )


    # GPU memory

    if torch.cuda.is_available():

        allocated = (

            torch.cuda.memory_allocated()
            / 1024**3

        )

        reserved = (

            torch.cuda.memory_reserved()
            / 1024**3

        )


        print(

            f"GPU Memory: "
            f"{allocated:.2f} GB allocated / "
            f"{reserved:.2f} GB reserved"

        )


# ============================================================
# 24. TRAINING COMPLETED
# ============================================================

print(
    "\n========================================"
)

print(
    "TRAINING COMPLETED"
)

print(
    "========================================"
)


print(
    "Best Validation Accuracy:",
    f"{best_val_accuracy:.2f}%"
)


print(
    "Model saved at:",
    model_path
)


# ============================================================
# 25. LOAD BEST MODEL
# ============================================================

check_point = torch.load(

    model_path,

    map_location=device,

    weights_only=True

)


model.load_state_dict(
    check_point["model_state_dict"]
)


model = model.to(device)

model.eval()


# Use saved class names if available

if "class_names" in check_point:

    class_names = check_point[
        "class_names"
    ]


if "image_size" in check_point:

    image_size = check_point[
        "image_size"
    ]


print(
    "\nBest saved model loaded successfully."
)


# ============================================================
# 26. FINAL TEST
# ============================================================

test_loss, test_accuracy = (
    validate_model(

        model,

        test_loader,

        criterion

    )
)


print(
    "\n========== TEST RESULTS =========="
)

print(
    f"Test Loss     : "
    f"{test_loss:.4f}"
)

print(
    f"Test Accuracy : "
    f"{test_accuracy:.2f}%"
)


web test


In [ ]:

# ============================================================
# 27. WEBCAM TRANSFORM
# ============================================================

webcam_transform = transforms.Compose([

    transforms.Resize(
        (image_size, image_size)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]

    )
])


# ============================================================
# 28. GOOGLE COLAB WEBCAM JAVASCRIPT
# ============================================================

display(Javascript("""

window.cameraVideo = null;

window.cameraCanvas = null;

window.cameraStream = null;

window.cameraStopped = false;


window.startCamera = async function() {

    window.cameraStopped = false;


    // Create video

    if (!window.cameraVideo) {

        window.cameraVideo =
            document.createElement('video');


        window.cameraVideo.style.width =
            '640px';

        window.cameraVideo.style.height =
            '480px';

        window.cameraVideo.style.border =
            '3px solid black';


        window.cameraVideo.autoplay = true;

        window.cameraVideo.playsInline = true;


        document.body.appendChild(
            window.cameraVideo
        );

    }


    // Create canvas

    if (!window.cameraCanvas) {

        window.cameraCanvas =
            document.createElement('canvas');

        window.cameraCanvas.width =
            640;

        window.cameraCanvas.height =
            480;

    }


    // Ask webcam permission

    window.cameraStream =
        await navigator.mediaDevices
        .getUserMedia({

            video: true,

            audio: false

        });


    window.cameraVideo.srcObject =
        window.cameraStream;


    await window.cameraVideo.play();


    return "Camera started";

};


window.getCameraFrame = function() {

    if (!window.cameraVideo) {

        return "";

    }


    window.cameraCanvas.width =
        window.cameraVideo.videoWidth;


    window.cameraCanvas.height =
        window.cameraVideo.videoHeight;


    let context =
        window.cameraCanvas
        .getContext('2d');


    context.drawImage(

        window.cameraVideo,

        0,

        0,

        window.cameraCanvas.width,

        window.cameraCanvas.height

    );


    return window.cameraCanvas
        .toDataURL(
            'image/jpeg',
            0.7
        );

};


window.stopCamera = function() {

    window.cameraStopped = true;


    if (window.cameraStream) {

        let tracks =
            window.cameraStream
            .getTracks();


        tracks.forEach(
            function(track) {

                track.stop();

            }
        );

    }


    if (window.cameraVideo) {

        window.cameraVideo.srcObject =
            null;

    }


    return "Camera stopped";

};


window.isCameraStopped = function() {

    return window.cameraStopped;

};

"""))


# ============================================================
# 29. START WEBCAM
# ============================================================

print(
    "\n=========================================="
)

print(
    "STARTING WEBCAM"
)

print(
    "=========================================="
)

print(
    "Allow camera permission when browser asks."
)


camera_result = eval_js(
    "startCamera()"
)


print(
    camera_result
)


# ============================================================
# 30. GET WEBCAM FRAME
# ============================================================

def get_webcam_image():

    data = eval_js(
        "getCameraFrame()"
    )


    if not data:

        return None


    # Remove base64 header

    image_data = data.split(
        ","
    )[1]


    # Decode base64

    image_bytes = base64.b64decode(
        image_data
    )


    # Convert to numpy

    image_array = np.frombuffer(

        image_bytes,

        dtype=np.uint8

    )


    # Decode JPEG

    frame = cv2.imdecode(

        image_array,

        cv2.IMREAD_COLOR

    )


    return frame


# ============================================================
# 31. RESNET18 WEBCAM PREDICTION
# ============================================================

@torch.inference_mode()

def predict_webcam_frame(frame):


    # BGR -> RGB

    frame_rgb = cv2.cvtColor(

        frame,

        cv2.COLOR_BGR2RGB

    )


    # Convert to PIL

    image = Image.fromarray(
        frame_rgb
    )


    # Apply same preprocessing

    image_tensor = webcam_transform(
        image
    )


    # Add batch dimension

    image_tensor = (
        image_tensor.unsqueeze(0)
    )


    # Move to GPU

    image_tensor = (
        image_tensor.to(device)
    )


    # Prediction

    outputs = model(
        image_tensor
    )


    # Convert to probability

    probabilities = torch.softmax(

        outputs,

        dim=1

    )


    # Get highest probability

    confidence, predicted_index = (
        torch.max(
            probabilities,
            dim=1
        )
    )


    confidence = (
        confidence.item()
    )


    predicted_index = (
        predicted_index.item()
    )


    predicted_letter = (
        class_names[predicted_index]
    )


    return (
        predicted_letter,
        confidence
    )


# ============================================================
# 32. WEBCAM RECOGNITION SETTINGS
# ============================================================

# Minimum confidence

CONFIDENCE_THRESHOLD = 0.80


# Number of continuous frames
# required before accepting letter

STABLE_FRAMES = 5


# Number of low-confidence frames
# needed to reset same-letter detection

NEUTRAL_FRAMES = 8


# Current word

recognized_word = ""


# Current candidate

candidate_letter = None

candidate_count = 0


# Last accepted letter

last_accepted_letter = None


# Neutral frame counter

neutral_count = 0


# ============================================================
# 33. WEBCAM INSTRUCTIONS
# ============================================================

print(
    "\n=========================================="
)

print(
    "SIGN LANGUAGE WEBCAM RECOGNITION"
)

print(
    "=========================================="
)

print()
print(
    "Show one alphabet sign at a time."
)

print()
print(
    "Example:"
)

print(
    "H -> E -> L -> L -> O"
)

print()
print(
    "Expected result:"
)

print(
    "HELLO"
)

print()
print(
    "For repeated letters such as LL:"
)

print(
    "1. Show L"
)

print(
    "2. Remove/lower your hand briefly"
)

print(
    "3. Show L again"
)

print()
print(
    "Press Ctrl + M + I to interrupt"
)

print(
    "the Colab cell if you want to stop."
)

print(
    "=========================================="
)


# ============================================================
# 34. CONTINUOUS WEBCAM RECOGNITION
# ============================================================

try:

    while True:


        # Get frame

        frame = get_webcam_image()


        if frame is None:

            continue


        # Predict

        letter, confidence = (
            predict_webcam_frame(frame)
        )


        # ----------------------------------------------------
        # LOW CONFIDENCE
        # ----------------------------------------------------

        if confidence < CONFIDENCE_THRESHOLD:


            neutral_count += 1


            candidate_letter = None

            candidate_count = 0


            # Allow same letter again
            # after hand is removed

            if neutral_count >= NEUTRAL_FRAMES:

                last_accepted_letter = None


            print(

                f"\rNo clear sign | "

                f"Confidence: "
                f"{confidence:.2f} | "

                f"Word: "
                f"{recognized_word}",

                end=""

            )


            time.sleep(0.05)


            continue


        # ----------------------------------------------------
        # CLEAR SIGN
        # ----------------------------------------------------

        neutral_count = 0


        # Same candidate

        if letter == candidate_letter:

            candidate_count += 1


        else:

            candidate_letter = letter

            candidate_count = 1


        # ----------------------------------------------------
        # ACCEPT STABLE LETTER
        # ----------------------------------------------------

        if candidate_count >= STABLE_FRAMES:


            # Accept first letter
            # or different letter

            if (

                last_accepted_letter is None

                or

                letter != last_accepted_letter

            ):


                recognized_word += letter


                last_accepted_letter = letter


                print(
                    f"\n\nLETTER ACCEPTED: "
                    f"{letter}"
                )


                print(
                    f"CURRENT WORD: "
                    f"{recognized_word}"
                )


                # Reset candidate counter

                candidate_count = 0


        # ----------------------------------------------------
        # LIVE STATUS
        # ----------------------------------------------------

        print(

            f"\rPrediction: "
            f"{letter} | "

            f"Confidence: "
            f"{confidence:.2f} | "

            f"Word: "
            f"{recognized_word}",

            end=""

        )


        # Small delay

        time.sleep(0.05)


except KeyboardInterrupt:

    print(
        "\n\nRecognition stopped by user."
    )


finally:

    eval_js(
        "stopCamera()"
    )

    print(
        "\nCamera stopped."
    )


# ============================================================
# 35. FINAL WORD
# ============================================================

print(
    "\n=========================================="
)

print(
    "FINAL RECOGNITION RESULT"
)

print(
    "=========================================="
)


if recognized_word:

    print(
        "English Word:"
    )

    print(
        recognized_word
    )

else:

    print(
        "No word recognized."
    )


# ============================================================
# 36. ENGLISH -> TAMIL TRANSLATION
# ============================================================

if recognized_word:


    try:


        print(
            "\nTranslating English -> Tamil..."
        )


        translator = GoogleTranslator(

            source="en",

            target="ta"

        )


        tamil_translation = (
            translator.translate(
                recognized_word
            )
        )


        print(
            "\nTamil Translation:"
        )


        print(
            tamil_translation
        )


    except Exception as e:


        print(
            "\nTranslation failed."
        )


        print(
            "Error:",
            e
        )


# ============================================================
# 37. FINAL OUTPUT
# ============================================================

print(
    "\n=========================================="
)

print(
    "PROJECT COMPLETED"
)

print(
    "=========================================="
)